# 00 · Data review and preprocessing decisions

**Goal:** look at the data before modelling and decide, with evidence, how to preprocess and validate:
do we need K-fold (and what kind), normalisation (which one), what to do with missing values, duplicates and
skewed columns, and how different the synthetic training data is from Salud's real data.

**Data** (built in Step 1 by `salud-propensity build`): one row per patient as of a call date, with features known
*before* the call, call fields, and whether the patient paid within 30 days (`paid_30d`).

| Set | Use |
|---|---|
| `dev` — synthetic, earlier 80% by date | all experiments (cross-validation) |
| `holdout` — synthetic, latest 20% | one final test in notebook 11 |
| `real` — Salud's 58 usable rows | one final test in notebook 11; never used to choose |

*Privacy: real data appears only as aggregates in this notebook. Outputs are stripped before any commit.*

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from salud_pipeline.phase2.propensity import modeling as M
from salud_pipeline.phase2.propensity import dictionary as D
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(world=WORLD)
print(f"dev (synthetic, 80% for experiments): {len(dev):,} rows, {dev.guarantor_id.nunique():,} patients, pay rate {dev.paid_30d.mean():.2f}")
print(f"holdout (synthetic, 20% held back, used once in notebook 11): {len(holdout):,} rows")
print(f"real (usable, final test only): {len(real)} rows, pay rate {real.paid_30d.mean():.2f}")

## 0. Is the synthetic world realistic? (calibration against Salud's 84 calls)

Every statistic below is computed with the same code on Salud's data and on the synthetic world: how calls split
(connection, disposition, contact type, result, actions, payment on the call), what patients look like at the call
(accounts, balances, debt age, payment and insurance history), the outcome rates, and each feature's direction of
effect. A synthetic value **matches** when it falls inside the real data's 95% bootstrap interval.
(`salud-synth-calibrate --world <folder>` prints the same table.)

In [ ]:
from salud_pipeline.paths import PHASE2_WORLDS
from salud_pipeline.phase2.synthetic import calibrate as C
from salud_pipeline.phase2.propensity.sources import load_real, load_synthetic
cal = C.compare(load_real(), load_synthetic(PHASE2_WORLDS[WORLD]), n_boot=300)
print(f"World '{WORLD}': {int(cal.match.sum())}/{len(cal)} statistics inside the real 95% interval")
display(C.summary(cal))
cal.style.format(precision=3).apply(lambda r: ["background-color: #e8f5e9" if r.match else "background-color: #ffebee"] * len(r), axis=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, grp in zip(axes, ["call mix", "relationships"]):
    t = cal[cal.group == grp].reset_index(drop=True)
    y = np.arange(len(t))
    ax.hlines(y, t["real 95% low"], t["real 95% high"], color="#9ecae1", lw=6, label="real 95% interval")
    ax.plot(t.real, y, "o", c="#3182bd", label="real"); ax.plot(t.synthetic, y, "D", c="#e6550d", label="synthetic")
    ax.set_yticks(y); ax.set_yticklabels(t.statistic, fontsize=8); ax.invert_yaxis(); ax.set_title(grp); ax.legend(loc="lower right")
    if grp == "relationships": ax.axvline(.5, ls="--", c="grey")
plt.tight_layout(); plt.show()

## 1. Rows, patients and labels

In [ ]:
per_patient = dev.groupby("guarantor_id").size()
multi = dev.groupby("guarantor_id").paid_30d.agg(["mean", "size"]).query("size > 1")
summary = pd.DataFrame({
    "synthetic dev": [len(dev), dev.guarantor_id.nunique(), per_patient.mean(), dev.paid_30d.mean(),
                      ((multi["mean"] > 0) & (multi["mean"] < 1)).mean(), f"{dev.as_of.min():%Y-%m-%d} → {dev.as_of.max():%Y-%m-%d}"],
    "real": [len(real), real.guarantor_id.nunique(), real.groupby("guarantor_id").size().mean(), real.paid_30d.mean(),
             np.nan, f"{pd.to_datetime(real.as_of).min():%Y-%m-%d} → {pd.to_datetime(real.as_of).max():%Y-%m-%d}"],
}, index=["rows", "patients", "rows per patient", "pay rate (30 days)", "repeat patients with mixed outcomes", "dates"])
summary

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
per_patient.value_counts().sort_index().plot.bar(ax=ax[0], color="#4C72B0"); ax[0].set(title="Calls per patient (synthetic)", xlabel="calls", ylabel="patients")
pd.DataFrame({"synthetic": [dev.paid_30d.mean()], "real": [real.paid_30d.mean()]}).T.plot.barh(ax=ax[1], legend=False, color=["#55A868"])
ax[1].set(title="Pay rate within 30 days", xlim=(0, 1)); plt.tight_layout()

## 2. Feature profile: missing values, zeros, skew, outliers, signal

In [ ]:
def auc1(x, y):
    m = x.notna()
    if m.sum() < 5 or x[m].nunique() < 2 or len(set(y[m])) < 2: return np.nan
    from sklearn.metrics import roc_auc_score
    a = roc_auc_score(y[m], x[m]); return max(a, 1 - a)

feats = M.FS_ALL + M.POST_EXTRA
prof = pd.DataFrame([{
    "feature": f, "known before call": f in D.PRE_FEATURES, "distinct": dev[f].nunique(),
    "missing %": 100 * dev[f].isna().mean(), "zero %": 100 * (dev[f] == 0).mean(),
    "skew": dev[f].astype(float).skew(),
    "max / p99": dev[f].max() / dev[f].quantile(.99) if dev[f].nunique() > 2 and dev[f].quantile(.99) > 0 else np.nan,
    "AUC alone (synthetic)": auc1(dev[f].astype(float), dev.paid_30d.values),
    "AUC alone (real)": auc1(real[f].astype(float), real.paid_30d.values) if f in real else np.nan,
} for f in feats]).set_index("feature")
prof.style.format(precision=2).background_gradient(subset=["skew", "zero %", "AUC alone (synthetic)", "AUC alone (real)"], cmap="Blues")

In [ ]:
money = ["balance_at_t", "amount_paid_365d", "n_payments_365d", "talk_seconds"]
fig, ax = plt.subplots(2, 4, figsize=(15, 5.5))
for i, f in enumerate(money):
    sns.histplot(dev[f], bins=40, ax=ax[0, i], color="#C44E52"); ax[0, i].set_title(f"{f} (raw, skew {dev[f].skew():.1f})")
    sns.histplot(np.log1p(dev[f]), bins=40, ax=ax[1, i], color="#4C72B0"); ax[1, i].set_title(f"log1p (skew {np.log1p(dev[f]).skew():.1f})")
plt.tight_layout()

## 3. Duplicated information (correlation)

In [ ]:
corr = dev[M.FS_ALL].astype(float).corr(method="spearman")
plt.figure(figsize=(11, 9)); sns.heatmap(corr, cmap="vlag", center=0, square=True, cbar_kws={"shrink": .6}); plt.title("Spearman correlation, pre-call features"); plt.show()
c = corr.abs().to_numpy().copy(); np.fill_diagonal(c, 0)
pairs = [(corr.index[i], corr.columns[j], round(c[i, j], 3)) for i in range(len(c)) for j in range(i + 1, len(c)) if c[i, j] > 0.9]
pd.DataFrame(pairs, columns=["feature A", "feature B", "|rho|"])

In [ ]:
pd.Series({k: ", ".join(v) for k, v in M.DUPLICATES.items()}, name="dropped in FS_PRUNED (kept → dropped)").to_frame()

## 4. How different is the synthetic world from Salud's real data?

In [ ]:
cmp = pd.DataFrame({"synthetic median": dev[M.FS_ALL].median(), "real median": real[M.FS_ALL].median(),
                    "synthetic mean": dev[M.FS_ALL].mean(), "real mean": real[M.FS_ALL].mean()})
cmp["real / synthetic (mean)"] = cmp["real mean"] / cmp["synthetic mean"].replace(0, np.nan)
cmp.style.format(precision=2).background_gradient(subset=["real / synthetic (mean)"], cmap="Reds")

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.2))
for a, f in zip(ax, ["n_accounts", "balance_at_t", "n_adjustments_365d", "n_prior_contacts_90d"]):
    sns.kdeplot(np.log1p(dev[f]), ax=a, label="synthetic", fill=True, cut=0)
    sns.kdeplot(np.log1p(real[f]), ax=a, label="real", fill=True, cut=0, warn_singular=False)
    a.set_title(f"log1p({f})"); a.legend()
plt.tight_layout()

### Adversarial validation
Train a model to tell synthetic rows from real ones. AUC ≈ 0.5 would mean the worlds look alike; ≈ 1.0 means they
are easy to tell apart. The most useful features for telling them apart are the ones that differ most.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
adv_feats = M.FS_PRUNED
syn_s = dev.sample(len(real) * 5, random_state=M.SEED)
A = pd.concat([syn_s[adv_feats], real[adv_feats]]).astype(float)
is_real = np.r_[np.zeros(len(syn_s)), np.ones(len(real))]
clf = HistGradientBoostingClassifier(max_depth=3, random_state=M.SEED)
p = cross_val_predict(clf, A, is_real, cv=StratifiedKFold(5, shuffle=True, random_state=M.SEED), method="predict_proba")[:, 1]
print(f"Adversarial AUC (synthetic vs real): {roc_auc_score(is_real, p):.3f}")
imp = permutation_importance(clf.fit(A, is_real), A, is_real, scoring="roc_auc", n_repeats=20, random_state=M.SEED)
pd.Series(imp.importances_mean, index=adv_feats).sort_values(ascending=False).head(8).rename("drop in AUC when shuffled").to_frame()

## 5. How much do the hidden traits explain? (oracle)

In [ ]:
dev_o = pd.concat([dev, M.oracle_traits(dev, WORLD)], axis=1)
o = M.run_experiment("B-05", M.pipeline(M.preprocess("z"), M.logreg(C=1.0)), dev_o, ["ability", "willingness", "hardship"],
                     notebook="00", description="ORACLE: hidden traits (ability, willingness, hardship) — not observable", log=False)

## 6. Which synthetic training design?

Two ways to build synthetic training rows. Compare both with the same reference model and the oracle (a model on
the hidden traits, i.e. the best any model could do):

| Design | Rows | Matches |
|---|---|---|
| **call** — one row per call, 30 days after the call | 3,321 | the real data's shape (real rows are calls) |
| **fixed** — one row per patient with an open balance on 2026-04-01, 30 days after | 1,350 | how the ranking is used ("score everyone today") and where the synthetic effects were planted |

In [ ]:
fdev, fhold, _ = M.load_data(design="fixed", world=WORLD)
REF = lambda: M.pipeline(M.preprocess("log_z"), M.logreg(C=1.0))
design_rows = []
for name, d in [("call", dev), ("fixed", fdev)]:
    d_o = pd.concat([d, M.oracle_traits(d, WORLD)], axis=1)
    sp = M.cv_splits(d)
    lr = M.run_experiment(f"D-{name.upper()}-LR", REF(), d, "FS_TRANSFER", notebook="00", splits=sp,
                          description=f"{name} design: L2 logistic regression, log1p + z, FS_TRANSFER")
    orc = M.run_experiment(f"D-{name.upper()}-ORACLE", M.pipeline(M.preprocess("z"), M.logreg(C=1.0)), d_o,
                           ["ability", "willingness", "hardship"], notebook="00", splits=sp,
                           description=f"{name} design: ORACLE hidden traits")
    design_rows.append({"design": name, "rows": len(d), "pay rate": d.paid_30d.mean(),
                        "model AUC": lr.auc.mean(), "oracle AUC (hidden traits)": orc.auc.mean(),
                        "model / oracle": (lr.auc.mean() - .5) / (orc.auc.mean() - .5)})
pd.DataFrame(design_rows).set_index("design")

## 7. Decisions

In [ ]:
decisions = pd.DataFrame([
    ("Validation", "Repeated (5×5) stratified K-fold, GROUPED by patient", "patients repeat; repeat outcomes differ; 4k rows → ~830 per test fold"),
    ("Holdout", "20% held back, used once: latest by date (call design) or a stratified random 20% of patients (fixed design: one date)", "an untouched final check"),
    ("Real data", "Final test only, with bootstrap CIs", f"{len(real)} rows → AUC standard error ≈ 0.07, too few to choose with"),
    ("Imbalance", "Stratify folds; no SMOTE; test balanced weights once", "70/30 and 50/50 are mild"),
    ("Missing values", "'Days since' → 'never' value + flag", "missing means 'never happened'"),
    ("Skew", "log1p on money/count columns for linear, distance, neural models", "skew 6–34, long tails"),
    ("Scaling", "Compare z, robust, quantile, WoE in notebook 02; none for trees", "which is best is an empirical question"),
    ("Duplicates", "FS_PRUNED keeps one per |rho|>0.9 cluster; also per-fold pruning", "copies split linear weights, confuse L1"),
    ("Synthetic ≠ real", "FS_TRANSFER drops prior-contact features; prefer scale-free transforms", "adversarial validation (section 4); the real extract has no contact history"),
    ("Training design", "Pre-call: fixed-date rows; post-call: call rows", "design comparison (section 6)"),
], columns=["topic", "decision", "evidence"])
decisions

In [ ]:
pd.DataFrame({k: [len(v), ", ".join(v)] for k, v in M.FEATURE_SETS.items()}, index=["n features", "features"]).T

## Findings (Salud-tuned world)
- **Calibration:** the Salud-tuned world matches **50 of 51** statistics against the real 95% intervals: all 25 call-mix, 2 outcome and 11 relationship statistics.
  The one miss is "adjustment in last 365 days": synthetic 25.0% against a real upper bound of 24.2%, a borderline miss.
- **Data:**
  - call design: 7,426 dev rows, 1,787 patients, pay rate 0.44;
  - fixed design: 1,435 patients, pay rate 0.35;
  - real: 58 usable rows, pay rate 0.50.
- **Adversarial AUC 0.945** (synthetic vs real), driven almost entirely by `days_since_last_contact` (shuffling it drops the AUC by 0.23).
  The real extract has no contact history, which is why the contact features are excluded from FS_TRANSFER. Without them, the remaining features are close.
- **The "oracle ceiling" is no longer a ceiling.** In this world, payment is driven mainly by an observable payment habit, not by the hidden traits.
  So the traits alone reach only 0.57 (call design) and 0.60 (fixed design), while ordinary models reach 0.87. Treat B-05 as "what the hidden traits explain", not as an upper bound.
- **Design:** the fixed design (one row per patient as of 2026-04-01) scores **0.867**, against **0.684** for the call design, with the same model.
  The models below use the fixed design.
- **Preprocessing decisions** carry over unchanged: grouped K-fold, log1p for skewed columns, a "never" fill for recency, pruning of duplicate columns.